# MLOps ДЗ №3 — Анализ качества датасета мошеннических транзакций

Этот ноутбук запускается на мастер-узле кластера Yandex Data Processing.

**Цель:** выявить минимум 3 типа проблем с данными перед их очисткой.

In [ ]:
# === Параметры ===
BUCKET = "mlops-otus-hw3-edit-2026"   # ЗАМЕНИТЕ на ваш бакет
INPUT_PATH  = f"s3a://{BUCKET}/raw/"
OUTPUT_PATH = f"s3a://{BUCKET}/cleaned/fraud_transactions/"

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, DoubleType,
)

spark = (
    SparkSession.builder
    .appName("MLOpsHW3-Analysis")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark

In [ ]:
schema = StructType([
    StructField("step",            IntegerType(), True),
    StructField("type",            StringType(),  True),
    StructField("amount",          DoubleType(),  True),
    StructField("nameOrig",        StringType(),  True),
    StructField("oldbalanceOrg",   DoubleType(),  True),
    StructField("newbalanceOrig",  DoubleType(),  True),
    StructField("nameDest",        StringType(),  True),
    StructField("oldbalanceDest",  DoubleType(),  True),
    StructField("newbalanceDest",  DoubleType(),  True),
    StructField("isFraud",         IntegerType(), True),
    StructField("isFlaggedFraud",  IntegerType(), True),
])

# Если у вас CSV — оставьте .csv, если parquet — поменяйте на .parquet(INPUT_PATH)
df = (
    spark.read
    .option("header", "true")
    .option("mode", "PERMISSIVE")
    .schema(schema)
    .csv(INPUT_PATH)
)
df.cache()
total = df.count()
print(f"Всего записей: {total:,}")
df.printSchema()

## Проблема №1 — пропущенные значения (NULL)

In [ ]:
null_counts = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns
])
null_counts.show(vertical=True)

## Проблема №2 — дубликаты

In [ ]:
dups = total - df.dropDuplicates().count()
print(f"Дубликатов: {dups:,}  ({100*dups/total:.4f}% от общего числа)")

## Проблема №3 — невалидные значения

- Отрицательные суммы.
- Неизвестные значения `type`.
- Отрицательные балансы.

In [ ]:
VALID_TYPES = ["PAYMENT", "TRANSFER", "CASH_OUT", "DEBIT", "CASH_IN"]

neg_amount        = df.filter(F.col("amount") < 0).count()
neg_balance_orig  = df.filter((F.col("oldbalanceOrg") < 0) | (F.col("newbalanceOrig") < 0)).count()
neg_balance_dest  = df.filter((F.col("oldbalanceDest") < 0) | (F.col("newbalanceDest") < 0)).count()
bad_type          = df.filter(~F.col("type").isin(VALID_TYPES)).count()

print(f"Отрицательная сумма (amount<0):  {neg_amount:,}")
print(f"Отрицательный баланс отправителя: {neg_balance_orig:,}")
print(f"Отрицательный баланс получателя:  {neg_balance_dest:,}")
print(f"Невалидный type:                  {bad_type:,}")

df.groupBy("type").count().orderBy(F.desc("count")).show()

## Проблема №4 — нарушение балансового тождества

Для исходящих транзакций (`PAYMENT`, `TRANSFER`, `CASH_OUT`, `DEBIT`):
$$oldbalanceOrg - newbalanceOrig = amount$$

Для `CASH_IN`:
$$newbalanceOrig - oldbalanceOrg = amount$$

In [ ]:
TOL = 0.01

is_outflow = F.col("type").isin(["PAYMENT", "TRANSFER", "CASH_OUT", "DEBIT"])
is_inflow  = F.col("type") == "CASH_IN"
merchant   = F.col("nameOrig").startswith("M")

diff_out = F.col("oldbalanceOrg") - F.col("newbalanceOrig")
diff_in  = F.col("newbalanceOrig") - F.col("oldbalanceOrg")

valid_out = is_outflow & (F.abs(diff_out - F.col("amount")) <= TOL)
valid_in  = is_inflow  & (F.abs(diff_in  - F.col("amount")) <= TOL)

balance_mismatch_count = df.filter(~(valid_out | valid_in | merchant)).count()
print(f"Строк с нарушением баланса: {balance_mismatch_count:,}  ({100*balance_mismatch_count/total:.2f}%)")

df.filter(~(valid_out | valid_in | merchant)).show(5, truncate=False)

## Сводный отчёт о найденных проблемах

In [ ]:
print("=" * 60)
print(f"Всего записей:                     {total:,}")
print(f"NULL в ключевых колонках:          см. таблицу выше")
print(f"Дубликаты:                         {dups:,}")
print(f"Невалидные значения:               см. блок №3")
print(f"Нарушение баланса:                 {balance_mismatch_count:,}")
print("=" * 60)
print("Найдено >= 4 типов проблем — переходим к скрипту очистки.")

## Запуск очистки прямо из ноутбука (опционально)

Производственный запуск — через `yc dataproc job create-pyspark`, см. README.
Здесь — для проверки логики:

In [ ]:
# Импортируем функции из скрипта (загрузите clean_data.py на мастер-узел)
import sys
sys.path.insert(0, "/home/ubuntu")  # путь, куда положили clean_data.py
from clean_data import clean  # noqa

cleaned = clean(df)
cleaned.write.mode("overwrite").option("compression", "snappy").parquet(OUTPUT_PATH)
print("OK, parquet сохранён в", OUTPUT_PATH)

In [ ]:
spark.stop()